In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

#Exercice 1 : Détection et suppression des doublons
train_df   = pd.read_csv('train.csv')
test_df    = pd.read_csv('test.csv')
gender_sub = pd.read_csv('gender_submission.csv')   # labels du test

# Fusion test + labels de référence
test_df = test_df.merge(gender_sub, on='PassengerId', how='left')

print("Shape train_df :", train_df.shape)
print("Shape test_df  :", test_df.shape, " ← inclut la colonne 'Survived' de gender_submission")
print()
print("Colonnes test_df :", test_df.columns.tolist())
test_df.head()


In [ ]:
for name, df in [('train_df', train_df), ('test_df', test_df)]:
    before = len(df)
    n_dup  = df.duplicated().sum()
    print(f"[{name}] Lignes avant : {before} | Doublons : {n_dup}")

# Suppression
train_df = train_df.drop_duplicates().reset_index(drop=True)
test_df  = test_df.drop_duplicates().reset_index(drop=True)

print()
print(f"train_df après suppression : {len(train_df)} lignes")
print(f"test_df  après suppression : {len(test_df)} lignes")


In [ ]:
                        #Exercice 2 : Gestion des valeurs manquantes
def missing_report(df, name):
    m = df.isnull().sum()
    pct = (m / len(df) * 100).round(2)
    report = pd.DataFrame({'Manquant': m, '% Manquant': pct})
    print(f"\n[{name}]")
    print(report[report['Manquant'] > 0])

missing_report(train_df, 'train_df')
missing_report(test_df,  'test_df')


In [ ]:
def impute_df(df, median_age=None, mode_embarked=None, mode_fare=None):
    """Applique les mêmes stratégies d'imputation sur train et test."""
    # Age  → médiane
    if median_age is None:
        median_age = df['Age'].median()
    df['Age'] = df['Age'].fillna(median_age)

    # Embarked → mode
    if mode_embarked is None:
        mode_embarked = df['Embarked'].mode()[0]
    df['Embarked'] = df['Embarked'].fillna(mode_embarked)

    # Fare → médiane (surtout utile pour test)
    if mode_fare is None:
        mode_fare = df['Fare'].median()
    df['Fare'] = df['Fare'].fillna(mode_fare)

    # Cabin → 'Unknown'
    df['Cabin'] = df['Cabin'].fillna('Unknown')

    return df, median_age, mode_embarked, mode_fare

# Fit sur train, apply sur les deux
train_df, med_age, mod_emb, med_fare = impute_df(train_df)
test_df, _, _, _                     = impute_df(test_df, med_age, mod_emb, med_fare)

print("Valeurs manquantes restantes - train :", train_df.isnull().sum().sum())
print("Valeurs manquantes restantes - test  :", test_df.isnull().sum().sum())


In [ ]:
                            # Exercice 3 : Ingénierie des fonctionnalités
def feature_engineering(df):
    # FamilySize
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    # Title extrait du nom
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    rare = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
    df['Title'] = df['Title'].replace(rare, 'Rare')
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    return df

train_df = feature_engineering(train_df)
test_df  = feature_engineering(test_df)

# LabelEncoder commun (fit sur train + test pour couvrir toutes les modalités)
le = LabelEncoder()
all_titles = pd.concat([train_df['Title'], test_df['Title']])
le.fit(all_titles)
train_df['Title_encoded'] = le.transform(train_df['Title'])
test_df['Title_encoded']  = le.transform(test_df['Title'])

print("Titres (train) :", train_df['Title'].value_counts().to_dict())
print("Titres (test)  :", test_df['Title'].value_counts().to_dict())
print("FamilySize - train head :", train_df['FamilySize'].head().tolist())


In [ ]:
                        #Exercice 4 : Détection et traitement des valeurs aberrantes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(axes, ['Age', 'Fare'], ['lightblue', 'lightcoral']):
    ax.boxplot(train_df[col], patch_artist=True, boxprops=dict(facecolor=color))
    ax.set_title(f'Boxplot Train - {col}')
plt.tight_layout(); plt.show()


In [ ]:
# Calcul des seuils sur le train uniquement
cap_fare = train_df['Fare'].quantile(0.98)
print(f"Plafond Fare (q98 train) : {cap_fare:.2f}")

for df, name in [(train_df, 'train'), (test_df, 'test')]:
    df['Fare_capped'] = df['Fare'].clip(upper=cap_fare)
    df['Fare_log']    = np.log1p(df['Fare'])
    print(f"[{name}] Fare_capped max : {df['Fare_capped'].max():.2f} | Fare_log skew : {df['Fare_log'].skew():.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color, title in zip(
    axes,
    ['Fare', 'Fare_capped', 'Fare_log'],
    ['steelblue', 'orange', 'green'],
    ['Fare - Original', 'Fare - Plafonné (q98)', 'Fare - Log']
):
    axes[list(axes).index(ax)].hist(train_df[col], bins=40, color=color, edgecolor='white')
    ax.set_title(title)
plt.tight_layout(); plt.show()


In [ ]:
                            #Exercice 5 : Standardisation et normalisation des données
std_cols = ['Age', 'FamilySize']
mm_cols  = ['Fare_capped', 'Fare_log']

# Fit sur train, transform sur train ET test
scaler_std = StandardScaler()
scaler_mm  = MinMaxScaler()

train_df[[c + '_std' for c in std_cols]] = scaler_std.fit_transform(train_df[std_cols])
test_df[[c + '_std'  for c in std_cols]] = scaler_std.transform(test_df[std_cols])

train_df[[c + '_mm' for c in mm_cols]] = scaler_mm.fit_transform(train_df[mm_cols])
test_df[[c + '_mm'  for c in mm_cols]] = scaler_mm.transform(test_df[mm_cols])

print("StandardScaler - Age_std train  : mean={:.4f}, std={:.4f}".format(
    train_df['Age_std'].mean(), train_df['Age_std'].std()))
print("MinMaxScaler   - Fare_capped_mm : min={:.4f}, max={:.4f}".format(
    train_df['Fare_capped_mm'].min(), train_df['Fare_capped_mm'].max()))


In [ ]:
                               #Exercice 6 : Encodage des caractéristiques
cat_cols = ['Sex', 'Embarked', 'Title']

# Encoder train et test ensemble pour avoir les mêmes colonnes
combined = pd.concat([train_df[cat_cols].assign(_split='train'),
                      test_df[cat_cols].assign(_split='test')])
dummies  = pd.get_dummies(combined[cat_cols], prefix=cat_cols)
dummies['_split'] = combined['_split'].values

train_dummies = dummies[dummies['_split'] == 'train'].drop('_split', axis=1).reset_index(drop=True)
test_dummies  = dummies[dummies['_split'] == 'test'].drop('_split', axis=1).reset_index(drop=True)

train_df = pd.concat([train_df, train_dummies], axis=1)
test_df  = pd.concat([test_df,  test_dummies],  axis=1)

encoded_cols = train_dummies.columns.tolist()
print("Colonnes one-hot créées :", encoded_cols)
print("Shape train :", train_df.shape, "| Shape test :", test_df.shape)


In [ ]:
                            #Exercice 7 : Transformation des données pour la caractéristique Âge
bins   = [0, 12, 18, 60, 100]
labels = ['Enfant', 'Adolescent', 'Adulte', 'Senior']

for df, name in [(train_df, 'train'), (test_df, 'test')]:
    df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)
    age_dummies    = pd.get_dummies(df['AgeGroup'], prefix='AgeGroup')
    # Ajouter les colonnes manquantes si nécessaire (cohérence train/test)
    for col in [f'AgeGroup_{l}' for l in labels]:
        if col not in age_dummies.columns:
            age_dummies[col] = 0
    globals()[f'{name}_df'] if False else None
    df[age_dummies.columns] = age_dummies.values
    print(f"[{name}] Distribution AgeGroup :")
    print(df['AgeGroup'].value_counts().sort_index().to_dict())
    print()


In [ ]:
print("=== RÉSUMÉ FINAL ===")
print(f"train_df : {train_df.shape}")
print(f"test_df  : {test_df.shape}  (contient 'Survived' de gender_submission)")
print()
new_cols = [c for c in train_df.columns if c not in
            ['PassengerId','Survived','Pclass','Name','Sex','Age','SibSp',
             'Parch','Ticket','Fare','Cabin','Embarked']]
print("Nouvelles colonnes créées :")
for c in new_cols:
    print(f"  {c}")
